### RUN in TPU ~01:35:00

- v24 added seed name .pkl file  

In [4]:
# CELL 0: Set Run Parameters
import os

os.environ["CACHE_LOOKBACK"] = "41"
os.environ["CACHE_START_DATE"] = "1998-01-01"
# Set to "True" to force identical splits regardless of size, or "False" to split dynamically
# os.environ["DEBUG_MODE"] = "False"

In [5]:
# CELL 1
import time

start_time = time.time()

In [6]:
# CELL 2: Setup
import sys, os
from pathlib import Path

RUNNING_IN_COLAB = "google.colab" in sys.modules
if RUNNING_IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    rl_root = Path(
        "/content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2"
    )
else:
    rl_root = Path("C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR_v2")

if str(rl_root) not in sys.path:
    sys.path.append(str(rl_root))
os.chdir(rl_root)

Mounted at /content/drive


In [7]:
# CELL 3: Imports
import torch
import numpy as np
import pandas as pd
from core.settings import TradingConfig, CacheConfig
from core.paths import OUTPUT_DIR, LOCAL_DATA_DIR
from data_pipeline.loader import load_processed_data
from data_pipeline.utils import get_master_trading_calendar, get_chronological_splits
from data_pipeline.screener import UniverseScreener
from rl_discovery.oracle import RLOracle
from rl_discovery.environment import DiscoveryEnv
from rl_discovery.adapter import RLVRGymEnv
from rl_discovery.agent import AbsoluteZeroAgent
from rl_discovery.trainer import RolloutBuffer, PPOTrainer
from rl_discovery.validator import AgentEvaluator

NOTEBOOKS_RLVR_ROOT: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2
PROJECT_ROOT: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks
GLOBAL_DATA_DIR: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/data
GLOBAL_PROCESSED_DIR: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/data/processed
LOCAL_DATA_DIR: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2/data
OUTPUT_DIR: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2/output



In [8]:
# CELL 4: Load Data & Environment Splits
print("Loading preprocessed data & AlphaCache...")
df_ohlcv, macro_df, features_df = load_processed_data()
config = TradingConfig()
df_close = df_ohlcv["Adj Close"].unstack(level=0).sort_index()
trading_calendar = get_master_trading_calendar(df_ohlcv, config.calendar_ticker)

screener = UniverseScreener(
    df_close=df_close,
    features_df=features_df,
    macro_df=macro_df,
    trading_calendar=trading_calendar,
    config=config,
)

cache_file_path = LOCAL_DATA_DIR / CacheConfig.get_filename()
print(f"cache_file_path:\n{cache_file_path}\n")
feature_cube = pd.read_parquet(cache_file_path)

oracle = RLOracle(screener, config)
oracle.precompute_reward_matrix(holding_period=config.holding_period)

# -------------------------------------------------------------------
# EXPLICIT TRAIN / VAL / TEST SPLITS (Custom Market Cycle Alignment)
# -------------------------------------------------------------------
# Ensure we only use dates that actually exist in our pre-processed AlphaCache
valid_dates = feature_cube.index.get_level_values("Date").unique()
valid_calendar = trading_calendar[trading_calendar.isin(valid_dates)]

# Define exact boundaries based on market cycle history
VAL_START = pd.Timestamp("2016-01-01")
TEST_START = pd.Timestamp("2022-04-01")

# 1. Raw Slicing
cal_train_raw = valid_calendar[valid_calendar < VAL_START]
cal_val_raw = valid_calendar[
    (valid_calendar >= VAL_START) & (valid_calendar < TEST_START)
]
cal_test_raw = valid_calendar[valid_calendar >= TEST_START]

# 2. Prevent Data Leakage (Holding Period Buffer)
# We drop the last `holding_period` days from Train and Val so active
# trades do not overlap into the next split's starting period.
hp = config.holding_period
cal_train = cal_train_raw[:-hp] if len(cal_train_raw) > hp else cal_train_raw
cal_val = cal_val_raw[:-hp] if len(cal_val_raw) > hp else cal_val_raw
cal_test = (
    cal_test_raw  # Test set doesn't need an end buffer (env logic handles stopping)
)

print(f"Custom splits applied! Leakage buffer dropped: {hp} days.")
# -------------------------------------------------------------------


# -------------------------------------------------------------
# FIXED ENVIRONMENT INITIALIZATIONS (Passing macro_df & config)
# -------------------------------------------------------------

env_train = DiscoveryEnv(
    feature_cube=feature_cube,
    reward_matrix=oracle.reward_matrix,
    calendar=cal_train,
    macro_df=macro_df,
    config=config,
)
gym_train = RLVRGymEnv(env_train, macro_df)

env_val = DiscoveryEnv(
    feature_cube=feature_cube,
    reward_matrix=oracle.reward_matrix,
    calendar=cal_val,
    macro_df=macro_df,
    config=config,
)
gym_val = RLVRGymEnv(env_val, macro_df)

env_test = DiscoveryEnv(
    feature_cube=feature_cube,
    reward_matrix=oracle.reward_matrix,
    calendar=cal_test,
    macro_df=macro_df,
    config=config,
)
gym_test = RLVRGymEnv(env_test, macro_df)

# ---> NEW: Freeze the scaler for Validation and Test environments # <---
gym_val.is_training = False
gym_test.is_training = False
# <---

Loading preprocessed data & AlphaCache...
cache_file_path:
/content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2/data/alpha_cache_41d_1998.parquet

Custom splits applied! Leakage buffer dropped: 5 days.


In [9]:
print("Parquet columns count:", len(feature_cube.columns))
print("Parquet columns:", list(feature_cube.columns))

Parquet columns count: 12
Parquet columns: ['41d_Log Price Gain', '41d_Sharpe (TRP)', '41d_Momentum (21d)', '41d_Info Ratio (63d)', '41d_Oversold (-RSI)', '41d_Dip Buyer (-dd_21)', '41d_Range Position (20d)', '41d_Return Autocorr (15d)', '41d_Low Volatility (-ATRP)', '41d_Slope_P_5_Z', '41d_Slope_V_5_Z', '41d_Convexity']


In [10]:
# CELL 5
# Calculate the duration in calendar years
train_years = (cal_train.max() - cal_train.min()).days / 365.25
val_years = (cal_val.max() - cal_val.min()).days / 365.25
test_years = (cal_test.max() - cal_test.min()).days / 365.25

print(f"Cal Train Start Date: {cal_train.min()}")
print(f"Cal Train End Date: {cal_train.max()}")
print(f"Cal Train Duration: {train_years:.2f} years")
print(f"Cal Train Trading Days: {len(cal_train)}\n")

print(f"Cal Val Start Date: {cal_val.min()}")
print(f"Cal Val End Date: {cal_val.max()}")
print(f"Cal Val Duration: {val_years:.2f} years")
print(f"Cal Val Trading Days: {len(cal_val)}\n")

print(f"Cal Test Start Date: {cal_test.min()}")
print(f"Cal Test End Date: {cal_test.max()}")
print(f"Cal Test Duration: {test_years:.2f} years")
print(f"Cal Test Trading Days: {len(cal_test)}")

Cal Train Start Date: 1998-01-02 00:00:00
Cal Train End Date: 2015-12-23 00:00:00
Cal Train Duration: 17.97 years
Cal Train Trading Days: 4524

Cal Val Start Date: 2016-01-04 00:00:00
Cal Val End Date: 2022-03-24 00:00:00
Cal Val Duration: 6.22 years
Cal Val Trading Days: 1568

Cal Test Start Date: 2022-04-01 00:00:00
Cal Test End Date: 2026-07-22 00:00:00
Cal Test Duration: 4.31 years
Cal Test Trading Days: 1079


### TODO: LR decay turned off

In [11]:
# CELL 6: Grid Setup & Environment Factory
import copy
import gymnasium as gym
from sklearn.model_selection import ParameterGrid

param_grid_test = {
    "ENT_COEF": [0.01],
    "downside_penalty": [1.0],
    "LR": [3e-4],
    "NUM_EPOCHS": [3],  # Just 3 epochs to see if it finishes
    "NUM_STEPS": [128],  # Short rollouts
    "MINI_BATCH_SIZE": [256],
    "NUM_ENVS": [4],  # Light CPU load
    "EVAL_FREQ": [1],
    "PATIENCE": [10],
    "WARMUP_EPOCHS": [0],
    "SEEDS": [[987654]],
}

param_grid_sweep = {
    # Test 0.01 (curious) vs 0.001 (focused/exploitative)
    "ENT_COEF": [0.01, 0.001],
    # Test 1.0 (standard) vs 2.0 (heavily penalize drawdowns/underperformance)
    "downside_penalty": [1.0, 2.0],
    # Test 3e-4 (standard) vs 1e-4 (slower, more stable learning)
    "LR": [3e-4, 1e-4],
    "NUM_EPOCHS": [75],  # Increased to 75 to let it traverse the timeline ~4 times
    "NUM_STEPS": [512],  # 256 steps * 64 envs = 16,384 batch size
    "MINI_BATCH_SIZE": [4096],  # Massive GPU matrix multiplications
    "NUM_ENVS": [64],  # 64 Parallel universes
    "EVAL_FREQ": [3],  # Eval every 3 epochs to save a little time
    "PATIENCE": [15],  # Give it more room to learn before early stopping
    "WARMUP_EPOCHS": [5],
    "SEEDS": [[987654]],  # Keep a single seed for the grid sweep
}

param_grid_robustness = {
    "ENT_COEF": [0.001],
    "downside_penalty": [1.0],
    "LR": [3e-4],
    "NUM_EPOCHS": [75],
    "NUM_STEPS": [256],
    "MINI_BATCH_SIZE": [4096],
    "NUM_ENVS": [64],
    "EVAL_FREQ": [3],
    "PATIENCE": [15],
    "WARMUP_EPOCHS": [5],
    # Test 5 distinct random universes
    "SEEDS": [[111111], [222222], [333333], [444444], [555555]],
}
########################
########################
# grid = list(ParameterGrid(param_grid_test))
# grid = list(ParameterGrid(param_grid_sweep))
grid = list(ParameterGrid(param_grid_sweep))
print(f"Total Grid Combinations to test: {len(grid)}")
########################
########################


def create_fresh_environments(
    base_config, grid_params, df_close, features_df, macro_df, trading_calendar
):
    """
    Creates fresh environments and oracles.
    Crucial for applying new grid params (like downside_penalty which alters the Oracle reward matrix).
    """
    run_config = copy.deepcopy(base_config)

    if "downside_penalty" in grid_params:
        run_config.downside_penalty = grid_params["downside_penalty"]
    if "rank_max_offset" in grid_params:
        run_config.rank_max_offset = grid_params["rank_max_offset"]

    screener = UniverseScreener(
        df_close=df_close,
        features_df=features_df,
        macro_df=macro_df,
        trading_calendar=trading_calendar,
        config=run_config,
    )
    oracle = RLOracle(screener, run_config)
    oracle.precompute_reward_matrix(holding_period=run_config.holding_period)

    # ---> NEW: Return a "Thunk" so Gym can spin up N Parallel Training Envs <---
    def make_train_env():
        def thunk():
            e = DiscoveryEnv(
                feature_cube,
                oracle.reward_matrix,
                cal_train,
                macro_df,
                run_config,
                randomize_start=True,  # <--- [NEW] Turn on randomness
                episode_steps=grid_params[
                    "NUM_STEPS"
                ],  # <--- [NEW] Pass dynamic NUM_STEPS
            )
            return RLVRGymEnv(e, macro_df)

        return thunk

    # Validation and Test stay single (Sequential evaluation is fast and deterministic)
    env_val = DiscoveryEnv(
        feature_cube, oracle.reward_matrix, cal_val, macro_df, run_config
    )
    gym_val = RLVRGymEnv(env_val, macro_df)
    gym_val.is_training = False

    env_test = DiscoveryEnv(
        feature_cube, oracle.reward_matrix, cal_test, macro_df, run_config
    )
    gym_test = RLVRGymEnv(env_test, macro_df)
    gym_test.is_training = False

    return make_train_env, gym_val, gym_test, run_config

Total Grid Combinations to test: 8


In [12]:
# CELL 7: Training Function
import random
import copy
import gymnasium as gym
import torch
import numpy as np
from rl_discovery.adapter import ObservationScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def train_agent_for_grid(grid_params, make_train_env, gym_val, checkpoint_dir):
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    seeds = grid_params["SEEDS"]
    num_epochs = grid_params["NUM_EPOCHS"]
    warmup_epochs = grid_params["WARMUP_EPOCHS"]
    num_envs = grid_params["NUM_ENVS"]

    grid_best_sharpe = -np.inf
    grid_best_model_path = None
    grid_best_seed = None
    grid_best_history = []
    grid_best_state_dict = None

    for seed in seeds:
        print(f"\n   -> 🚀 SEED: {seed}")
        set_seed(seed)

        # [NEW] 1. OS-Aware Vectorization (Safe for Windows, Fast for Colab)
        if "google.colab" in sys.modules:
            # print("   -> ⚡ Colab detected: Using AsyncVectorEnv (Multiprocessing)")
            envs = gym.vector.AsyncVectorEnv(
                [make_train_env() for _ in range(num_envs)]
            )
        else:
            print("   -> 💻 Local detected: Using SyncVectorEnv (Single-core safe)")
            envs = gym.vector.SyncVectorEnv([make_train_env() for _ in range(num_envs)])

        obs_dim = envs.single_observation_space.shape[0]
        action_dim = envs.single_action_space.shape[0]

        agent = AbsoluteZeroAgent(obs_dim=obs_dim, action_dim=action_dim).to(device)
        trainer = PPOTrainer(
            agent,
            lr=grid_params["LR"],
            clip_coef=0.2,
            ent_coef=grid_params["ENT_COEF"],
        )

        buffer = RolloutBuffer(
            num_steps=grid_params["NUM_STEPS"],
            num_envs=num_envs,
            obs_dim=obs_dim,
            action_dim=action_dim,
            device=device,
        )

        seed_best_val_sharpe = -np.inf
        patience_counter = 0
        seed_history = []

        for epoch in range(num_epochs):
            obs, _ = envs.reset()
            epoch_rewards = []

            for step in range(grid_params["NUM_STEPS"]):
                obs_tensor = torch.tensor(obs, dtype=torch.float32).to(device)
                with torch.no_grad():
                    action, logprob, _, value = agent.get_action_and_value(obs_tensor)

                raw_action = action.cpu().numpy()
                next_obs, rewards, terminations, truncations, _ = envs.step(raw_action)

                dones = np.logical_or(terminations, truncations)

                buffer.add(obs, action, logprob, rewards, value, dones)
                # Mean reward across the 8 parallel envs for logging
                epoch_rewards.append(rewards.mean())

                obs = next_obs

            # PPO Update
            obs_tensor = torch.tensor(obs, dtype=torch.float32).to(device)
            next_done_tensor = torch.tensor(dones, dtype=torch.float32).to(device)

            with torch.no_grad():
                next_values = agent.get_value(obs_tensor)

            buffer.compute_advantages(next_values, next_done=next_done_tensor)

            # [NEW] 2. Uncommented Learning Rate Decay
            trainer.update_lr(epoch + 1, num_epochs)

            diagnostics = trainer.update(
                buffer, update_epochs=4, mini_batch_size=grid_params["MINI_BATCH_SIZE"]
            )
            buffer.step = 0

            diagnostics["epoch"] = epoch + 1
            diagnostics["avg_reward"] = float(np.mean(epoch_rewards))
            seed_history.append(diagnostics)

            if (epoch + 1) % grid_params["EVAL_FREQ"] == 0:
                # [NEW] 3. AsyncVectorEnv requires `get_attr` to access properties of sub-environments
                gym_val.scaler.load_state(envs.get_attr("scaler")[0])

                val_results = AgentEvaluator.evaluate(agent, gym_val, device)
                val_sharpe = val_results["sharpe_ratio"]

                is_warmup = (epoch + 1) <= warmup_epochs
                status_icon = "🔥 [WARMUP]" if is_warmup else ""

                print(
                    f"      Ep {epoch+1:03d} | Rew: {diagnostics['avg_reward']:.4f} | Loss: {diagnostics['total_loss']:.4f} | Val Sharpe: {val_sharpe:.3f} {status_icon}"
                )

                if not is_warmup:
                    if val_sharpe > seed_best_val_sharpe:
                        seed_best_val_sharpe = val_sharpe
                        patience_counter = 0

                        hp_tag = f"ent_{params['ENT_COEF']}_pen_{params['downside_penalty']}_lr_{params['LR']}"
                        dynamic_name = f"model_{hp_tag}_seed_{seed}_ep_{epoch+1}_sh_{val_sharpe:.2f}.pt"

                        seed_best_path = checkpoint_dir / dynamic_name

                        torch.save(agent.state_dict(), str(seed_best_path))
                        print(f"         🟢 NEW BEST! Saved.")

                        if val_sharpe > grid_best_sharpe:
                            grid_best_sharpe = val_sharpe
                            grid_best_model_path = seed_best_path
                            grid_best_seed = seed
                            grid_best_history = list(seed_history)
                            grid_best_state_dict = copy.deepcopy(agent.state_dict())
                    else:
                        patience_counter += 1
                        print(
                            f"         🔴 No improvement. Patience: {patience_counter}/{grid_params['PATIENCE']}"
                        )

                    if patience_counter >= grid_params["PATIENCE"]:
                        print(f"      🛑 EARLY STOPPING triggered for Seed {seed}")
                        break

        envs.close()

    return (
        grid_best_model_path,
        grid_best_sharpe,
        grid_best_seed,
        grid_best_history,
        grid_best_state_dict,
    )

In [13]:
# CELL 8: Grid Execution & OOS Evaluation
import pickle
import json

checkpoint_dir = OUTPUT_DIR / "model_checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

global_ultimate_sharpe = -np.inf
global_ultimate_pkl = None

print("\n" + "=" * 60)
print("🏆 STARTING PHASE 3: AUTOMATED HYPERPARAMETER SWEEP 🏆")
print("=" * 60)

for idx, params in enumerate(grid):
    # 1. DEFINE THE FILENAME ONCE (including seed)
    current_seed = params["SEEDS"][0]
    hp_tag = f"seed_{current_seed}_ent_{params['ENT_COEF']}_pen_{params['downside_penalty']}_lr_{params['LR']}"
    dynamic_pkl_name = f"oos_results_{hp_tag}.pkl"
    results_save_path = OUTPUT_DIR / dynamic_pkl_name

    # 2. CHECK FOR SKIP
    if results_save_path.exists():
        print(
            f"\n✅ [SKIPPING] GRID COMBO {idx+1}/{len(grid)} - Already completed: {dynamic_pkl_name}"
        )
        with open(results_save_path, "rb") as f:
            prev_results = pickle.load(f)
            if prev_results["sharpe_ratio"] > global_ultimate_sharpe:
                global_ultimate_sharpe = prev_results["sharpe_ratio"]
                global_ultimate_pkl = dynamic_pkl_name
        continue

    print(f"\n\n{'='*50}")
    print(f"🔧 GRID COMBO {idx+1}/{len(grid)}")
    print(json.dumps({k: v for k, v in params.items() if k != "SEEDS"}, indent=2))
    print(f"Target Filename: {dynamic_pkl_name}")
    print(f"{'='*50}")

    # Create Environments
    make_train_env, gym_val, gym_test, run_config = create_fresh_environments(
        config, params, df_close, features_df, macro_df, trading_calendar
    )

    # Train (Passing HP tag to the training function so it can name checkpoints properly)
    best_model_path, best_sharpe, best_seed, best_history, best_state_dict = (
        train_agent_for_grid(params, make_train_env, gym_val, checkpoint_dir)
    )

    if best_state_dict is None:
        print("⚠️ No model survived warmup. Skipping OOS.")
        continue

    print(
        f"\n⭐ [GRID {idx+1} WINNER] Seed: {best_seed} | Val Sharpe: {best_sharpe:.3f}"
    )

    # Execute Walk-Forward OOS Evaluation
    obs_dim = gym_test.observation_space.shape[0]
    action_dim = gym_test.action_space.shape[0]
    champion_agent = AbsoluteZeroAgent(obs_dim=obs_dim, action_dim=action_dim).to(
        device
    )
    champion_agent.load_state_dict(best_state_dict)

    gym_test.scaler.load_state(gym_val.scaler)

    metadata_payload = params.copy()
    metadata_payload["BEST_SEED"] = best_seed
    metadata_payload["CHAMPION_FILE"] = (
        best_model_path.name if best_model_path else "None"
    )

    results = AgentEvaluator.evaluate(
        champion_agent, gym_test, device, detailed_log=True, metadata=metadata_payload
    )
    results["metadata"] = metadata_payload
    results["training_history"] = best_history

    # 3. SAVE USING THE SAME FILENAME DEFINED AT TOP
    with open(results_save_path, "wb") as f:
        pickle.dump(results, f)

    print(
        f"   -> [OOS RESULTS] Return: {results['total_return']*100:.2f}% | Sharpe: {results['sharpe_ratio']:.3f}"
    )
    print(f"   -> 💾 Saved to: {dynamic_pkl_name}")

    if results["sharpe_ratio"] > global_ultimate_sharpe:
        global_ultimate_sharpe = results["sharpe_ratio"]
        global_ultimate_pkl = dynamic_pkl_name

print("\n" + "*" * 60)
print("🏆 SWEEP COMPLETE: GLOBAL CHAMPION CROWNED 🏆")
print(f"Best OOS Sharpe: {global_ultimate_sharpe:.3f}")
print(f"Data File:       {global_ultimate_pkl}")
print("*" * 60 + "\n")
print(f"Winning file saved to: {results_save_path}")


🏆 STARTING PHASE 3: AUTOMATED HYPERPARAMETER SWEEP 🏆


🔧 GRID COMBO 1/8
{
  "ENT_COEF": 0.01,
  "EVAL_FREQ": 3,
  "LR": 0.0003,
  "MINI_BATCH_SIZE": 4096,
  "NUM_ENVS": 64,
  "NUM_EPOCHS": 75,
  "NUM_STEPS": 512,
  "PATIENCE": 15,
  "WARMUP_EPOCHS": 5,
  "downside_penalty": 1.0
}
Target Filename: oos_results_seed_987654_ent_0.01_pen_1.0_lr_0.0003.pkl

   -> 🚀 SEED: 987654
      Ep 003 | Rew: 0.0014 | Loss: -0.1967 | Val Sharpe: 1.321 🔥 [WARMUP]
      Ep 006 | Rew: 0.0012 | Loss: -0.2022 | Val Sharpe: 1.199 
         🟢 NEW BEST! Saved.
      Ep 009 | Rew: 0.0017 | Loss: -0.2048 | Val Sharpe: 1.127 
         🔴 No improvement. Patience: 1/15
      Ep 012 | Rew: 0.0012 | Loss: -0.2060 | Val Sharpe: 1.494 
         🟢 NEW BEST! Saved.
      Ep 015 | Rew: 0.0011 | Loss: -0.2070 | Val Sharpe: 1.296 
         🔴 No improvement. Patience: 1/15
      Ep 018 | Rew: 0.0015 | Loss: -0.2077 | Val Sharpe: 1.225 
         🔴 No improvement. Patience: 2/15
      Ep 021 | Rew: 0.0021 | Loss: -0.2099 | Val

In [14]:
# CELL 9: Timing
end_time = time.time()
print(f"Time: {time.strftime('%H:%M:%S', time.gmtime(end_time - start_time))}")

Time: 02:34:23


In [15]:
# CELL 10: Disconnect from TPU
from google.colab import runtime

# This will restart and unassign the current runtime,
# effectively disconnecting from the T4 GPU.
runtime.unassign()
print("Runtime unassigned. This session has been disconnected.")

Runtime unassigned. This session has been disconnected.


In [16]:
# # CELL 11: Convert .pt checkpoints to .pkl OOS results
# def convert_checkpoint_to_oos(checkpoint_path, gym_test, gym_val_with_scaler, device):
#     # 1. Load weights
#     state_dict = torch.load(checkpoint_path, map_location=device)

#     # 2. Setup Agent
#     obs_dim = gym_test.observation_space.shape[0]
#     action_dim = gym_test.action_space.shape[0]
#     agent = AbsoluteZeroAgent(obs_dim=obs_dim, action_dim=action_dim).to(device)
#     agent.load_state_dict(state_dict)

#     # 3. CRITICAL: Sync the scaler from Validation (where it was frozen) to Test
#     gym_test.scaler.load_state(gym_val_with_scaler.scaler)

#     # 4. Evaluate
#     results = AgentEvaluator.evaluate(agent, gym_test, device, detailed_log=True)
#     return results

In [17]:
# # CELL 12: Extraction Logic: Finding the "Champion at Epoch 50"
# import re
# import pickle

# def get_champion_from_checkpoints(checkpoint_dir, max_epoch=50):
#     files = list(checkpoint_dir.glob("model_seed_*.pt"))
#     best_file = None
#     max_val_sharpe = -np.inf

#     # Regex to extract seed, epoch, and sharpe from: model_seed_111111_ep_6_sh_1.56.pt
#     pattern = re.compile(r"model_seed_(\d+)_ep_(\d+)_sh_([\d\.]+).pt")

#     for f in files:
#         match = pattern.search(f.name)
#         if match:
#             seed = int(match.group(1))
#             epoch = int(match.group(2))
#             val_sharpe = float(match.group(3))

#             # Filter by your desired epoch limit
#             if epoch <= max_epoch:
#                 if val_sharpe > max_val_sharpe:
#                     max_val_sharpe = val_sharpe
#                     best_file = f

#     return best_file, max_val_sharpe


# # Usage
# target_epoch = 50
# ckpt_path, v_sharpe = get_champion_from_checkpoints(
#     OUTPUT_DIR / "model_checkpoints", max_epoch=target_epoch
# )

# if ckpt_path:
#     print(
#         f"Found Best Model at Ep {target_epoch}: {ckpt_path.name} (Val Sharpe: {v_sharpe})"
#     )

#     # Now execute the evaluation (ensure gym_test and gym_val are initialized in your environment)
#     oos_results = convert_checkpoint_to_oos(ckpt_path, gym_test, gym_val, device)

#     # Save as .pkl
#     save_name = f"recovered_oos_ep{target_epoch}_sh{v_sharpe}.pkl"
#     with open(OUTPUT_DIR / save_name, "wb") as f:
#         pickle.dump(oos_results, f)
#     print(f"Saved OOS evaluation to {save_name}")
# else:
#     print("No checkpoints found matching criteria.")